# Week 8: Advanced RAG and Debugging

**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Implement query transformation techniques** — HyDE, multi-query, and query decomposition
- **Apply re-ranking with cross-encoders** — improve retrieval precision after initial bi-encoder search
- **Build multi-hop retrieval chains** — handle questions requiring information from multiple documents
- **Evaluate RAG with RAGAS-inspired metrics** — faithfulness, answer relevancy, context precision, context recall
- **Debug retrieval failures systematically** — diagnose whether problems stem from retrieval or generation

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers ragas langchain langchain-community chromadb

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import numpy as np
from transformers import pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
import warnings
warnings.filterwarnings('ignore')

### Sample Knowledge Base

We'll use a small corpus throughout this notebook. In production, you would replace this with your own documents.

In [ ]:
# Shared knowledge base for demos
KNOWLEDGE_BASE = [
    "The Eiffel Tower is located in Paris, France. It was built in 1889.",
    "Paris is the capital of France and a major European city.",
    "The Great Wall of China stretches across northern China.",
    "Python is popular for AI and machine learning.",
    "The human brain contains around 86 billion neurons.",
]
print("Knowledge base loaded:", len(KNOWLEDGE_BASE), "documents")

---
## 1. Query Transformation

**Query transformation** improves retrieval by modifying the query before search. Three key techniques:

| Technique | Idea | When to Use |
|-----------|------|-------------|
| **HyDE** | Generate a hypothetical answer, embed *that* for retrieval | Short queries, semantic gap |
| **Multi-query** | Rewrite question in 3+ ways, retrieve for each, merge | Ambiguous or broad questions |
| **Query decomposition** | Break complex questions into sub-questions | Multi-fact, compositional queries |

We use **Flan-T5** (open-source) for all transformations — no API key required.

### 1.1 HyDE (Hypothetical Document Embeddings)

Instead of embedding the raw query, we:
1. Generate a *hypothetical* answer to the query
2. Embed the hypothetical answer
3. Use that embedding for retrieval

This bridges the **lexical gap** — queries and documents often use different words for the same concept.

In [ ]:
# Load Flan-T5 for query transformation (no API key needed)
flan = pipeline("text2text-generation", model="google/flan-t5-base", max_length=128)

def generate_hypothetical_answer(query: str, num_hypotheses: int = 1) -> list[str]:
    """Generate hypothetical answers to use as HyDE embeddings."""
    prompt = f"Write a short answer to this question: {query}"
    outputs = flan(prompt, max_new_tokens=64, num_return_sequences=num_hypotheses, do_sample=True)
    return [o["generated_text"].strip() for o in outputs]

# Example
query = "Where is the Eiffel Tower located?"
hyde_docs = generate_hypothetical_answer(query)
print("Query:", query)
print("Hypothetical answer (for retrieval):", hyde_docs[0])

In [ ]:
# Build FAISS index and compare retrieval with vs without HyDE
knowledge_base = KNOWLEDGE_BASE

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = bi_encoder.encode(knowledge_base)
index = faiss.IndexFlatIP(kb_embeddings.shape[1])
faiss.normalize_L2(kb_embeddings)
index.add(kb_embeddings)

def retrieve(query_embedding, k=2):
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding, k)
    return [(knowledge_base[i], float(s)) for i, s in zip(indices[0], scores[0])]

# Standard retrieval (embed query directly)
query_emb = bi_encoder.encode([query])
standard_results = retrieve(query_emb)

# HyDE retrieval (embed hypothetical answer)
hyde_emb = bi_encoder.encode(hyde_docs)
hyde_results = retrieve(hyde_emb)

print("=== Standard Retrieval ===")
for doc, score in standard_results:
    print(f"  [{score:.3f}] {doc[:60]}...")
print("\n=== HyDE Retrieval ===")
for doc, score in hyde_results:
    print(f"  [{score:.3f}] {doc[:60]}...")

### 1.2 Multi-Query

Rewrite the question in multiple ways, retrieve for each, then merge and deduplicate results. Helps when the original phrasing doesn't match document vocabulary.

In [ ]:
def generate_multi_queries(query: str, n: int = 3) -> list[str]:
    """Generate n alternative phrasings of the query."""
    prompt = f"Rephrase this question in a different way: {query}"
    outputs = flan(prompt, max_new_tokens=64, num_return_sequences=n, do_sample=True)
    queries = [query] + [o["generated_text"].strip() for o in outputs]
    return list(dict.fromkeys(queries))  # dedupe while preserving order

def multi_query_retrieve(query: str, k_per_query: int = 2) -> list[tuple[str, float]]:
    """Retrieve using multiple query phrasings and merge by score."""
    queries = generate_multi_queries(query, n=3)
    all_results = []
    seen_docs = set()
    for q in queries:
        q_emb = bi_encoder.encode([q])
        for doc, score in retrieve(q_emb, k=k_per_query):
            if doc not in seen_docs:
                seen_docs.add(doc)
                all_results.append((doc, score))
            else:
                idx = next(i for i, (d, _) in enumerate(all_results) if d == doc)
                all_results[idx] = (doc, max(all_results[idx][1], score))
    all_results.sort(key=lambda x: -x[1])
    return all_results[:k_per_query * 2]

mq_results = multi_query_retrieve("What city has the Eiffel Tower?")
print("Multi-query retrieval:")
for doc, score in mq_results:
    print(f"  [{score:.3f}] {doc[:60]}...")

### 1.3 Query Decomposition

Break complex questions into sub-questions, retrieve for each, then combine contexts. Essential for multi-hop reasoning.

In [ ]:
def decompose_query(query: str) -> list[str]:
    """Break a complex question into sub-questions."""
    prompt = f"Split this into 2-3 simpler questions, one per line: {query}"
    out = flan(prompt, max_new_tokens=128)[0]["generated_text"].strip()
    subqs = [q.strip() for q in out.split("\n") if q.strip()]
    return subqs if subqs else [query]

def decomposed_retrieve(query: str, k_per_sub: int = 2) -> list[tuple[str, float]]:
    """Retrieve for each sub-question and merge."""
    subqs = decompose_query(query)
    all_results = []
    for sq in subqs:
        q_emb = bi_encoder.encode([sq])
        for doc, score in retrieve(q_emb, k=k_per_sub):
            all_results.append((doc, score))
    all_results.sort(key=lambda x: -x[1])
    seen = set()
    deduped = []
    for doc, score in all_results:
        if doc not in seen:
            seen.add(doc)
            deduped.append((doc, score))
    return deduped[:5]

complex_q = "What country is the Eiffel Tower in and when was it built?"
subqs = decompose_query(complex_q)
print("Sub-questions:", subqs)
print("\nDecomposed retrieval:")
for doc, score in decomposed_retrieve(complex_q):
    print(f"  [{score:.3f}] {doc[:60]}...")

---
## 2. Re-Ranking with Cross-Encoders

| Model Type | How it works | Speed | Accuracy |
|------------|--------------|-------|----------|
| **Bi-encoder** | Encode query and docs separately, compare via dot product | Fast | Good |
| **Cross-encoder** | Encode query+doc together, output relevance score | Slow | Better |

**Strategy**: Retrieve top-20 with bi-encoder (fast), then re-rank to top-5 with cross-encoder (accurate).

In [ ]:
# Load cross-encoder for re-ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, docs: list[str], top_k: int = 5) -> list[tuple[str, float]]:
    """Re-rank documents using cross-encoder."""
    pairs = [[query, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: -x[1])
    return ranked[:top_k]

# Retrieve top-20 with bi-encoder
query = "Where is the Eiffel Tower?"
q_emb = bi_encoder.encode([query])
faiss.normalize_L2(q_emb)
scores, indices = index.search(q_emb, 20)
candidates = [knowledge_base[i] for i in indices[0]]

# Re-rank to top-5
reranked = rerank(query, candidates, top_k=5)

print("=== Top 5 after Re-Ranking ===")
for i, (doc, score) in enumerate(reranked, 1):
    print(f"{i}. [{score:.3f}] {doc}")

In [ ]:
# Compare: bi-encoder top-5 vs cross-encoder re-ranked top-5
bi_top5 = [(knowledge_base[i], float(scores[0][i])) for i in indices[0][:5]]

print("Bi-encoder top-5 (by cosine similarity):")
for i, (doc, s) in enumerate(bi_top5, 1):
    print(f"  {i}. {doc[:55]}...")
print("\nCross-encoder re-ranked top-5:")
for i, (doc, s) in enumerate(reranked, 1):
    print(f"  {i}. {doc[:55]}...")

---
## 3. Multi-Hop Retrieval

Some questions require information from **multiple documents**. Single retrieval often misses one piece.

**Iterative retrieval**:
1. Retrieve for the original query
2. Generate an intermediate answer (or extract key entities)
3. Retrieve again using the intermediate result
4. Combine contexts and generate final answer

In [ ]:
# Extended knowledge base for multi-hop demo
multi_hop_kb = [
    "The Eiffel Tower is in Paris.",
    "Paris is the capital of France.",
    "France is in Europe.",
    "The Eiffel Tower was built in 1889 by Gustave Eiffel.",
    "Gustave Eiffel was a French engineer.",
    "The Great Wall is in China.",
    "China is in Asia.",
]

mh_emb = bi_encoder.encode(multi_hop_kb)
mh_index = faiss.IndexFlatIP(mh_emb.shape[1])
faiss.normalize_L2(mh_emb)
mh_index.add(mh_emb)

def mh_retrieve(query_emb, k=3):
    faiss.normalize_L2(query_emb)
    s, idx = mh_index.search(query_emb, k)
    return [(multi_hop_kb[i], float(s[0][j])) for j, i in enumerate(idx[0])]

# Multi-hop question: "What continent is the Eiffel Tower in?"
# Single hop might get "Eiffel Tower is in Paris" but not "Paris is in France" or "France is in Europe"
q1 = "What continent is the Eiffel Tower in?"
step1 = mh_retrieve(bi_encoder.encode([q1]))
print("Step 1 - Query:", q1)
print("Retrieved:", [d for d, _ in step1])

# Use first result to form follow-up query
intermediate = "Paris France continent"
step2 = mh_retrieve(bi_encoder.encode([intermediate]))
print("\nStep 2 - Follow-up:", intermediate)
print("Retrieved:", [d for d, _ in step2])

### MRR (Mean Reciprocal Rank)

MRR measures retrieval quality: for each query, we find the rank of the first relevant document. MRR = average of 1/rank across queries.

In [ ]:
def mean_reciprocal_rank(ranked_docs: list[str], relevant_doc: str) -> float:
    """MRR: 1/rank of first relevant doc, or 0 if not found."""
    for i, doc in enumerate(ranked_docs, 1):
        if relevant_doc in doc or doc in relevant_doc:
            return 1.0 / i
    return 0.0

# Example: for "Where is the Eiffel Tower?", the most relevant doc mentions Paris
relevant = "The Eiffel Tower is located in Paris"
mrr_bi = mean_reciprocal_rank([d for d, _ in bi_top5], relevant)
mrr_rerank = mean_reciprocal_rank([d for d, _ in reranked], relevant)
print(f"MRR (bi-encoder top-5): {mrr_bi:.3f}")
print(f"MRR (after re-ranking):  {mrr_rerank:.3f}")
print("Re-ranking improves MRR when it moves relevant docs higher.")

In [ ]:
def iterative_retrieve(query: str, max_steps: int = 2) -> list[str]:
    """Multi-hop: retrieve, generate intermediate, retrieve again."""
    all_docs = []
    current_query = query
    for step in range(max_steps):
        q_emb = bi_encoder.encode([current_query])
        results = mh_retrieve(q_emb, k=3)
        docs = [d for d, _ in results]
        all_docs.extend(docs)
        if step == 0 and docs:
            # Use LLM to extract key info for next hop
            context = " ".join(docs)
            prompt = f"Based on: {context}\nQuestion: {query}\nWhat key fact or entity should we search for next? One short phrase."
            next_phrase = flan(prompt, max_new_tokens=20)[0]["generated_text"].strip()
            current_query = next_phrase if next_phrase else current_query
    return list(dict.fromkeys(all_docs))

final_docs = iterative_retrieve("What continent is the Eiffel Tower in?")
print("Combined contexts for multi-hop:")
for d in final_docs:
    print(" -", d)

---
## 4. RAG Evaluation

Key metrics (RAGAS-inspired):

| Metric | Meaning |
|--------|--------|
| **Faithfulness** | Answer is grounded in the retrieved context |
| **Answer relevancy** | Answer addresses the question |
| **Context precision** | Retrieved contexts are relevant |
| **Context recall** | All needed information was retrieved |

We build a **simple evaluation framework** that works without OpenAI — using overlap, heuristics, and manual scoring.

### Combined Pipeline: HyDE + Re-Ranking

We can chain techniques: use HyDE for retrieval, then re-rank the results with a cross-encoder for best precision.

In [ ]:
def hyde_rerank_pipeline(query: str, k_retrieve: int = 10, k_final: int = 3) -> list[tuple[str, float]]:
    """Retrieve with HyDE, then re-rank with cross-encoder."""
    hyde_docs = generate_hypothetical_answer(query)
    hyde_emb = bi_encoder.encode(hyde_docs)
    faiss.normalize_L2(hyde_emb)
    scores, indices = index.search(hyde_emb, k_retrieve)
    candidates = [knowledge_base[i] for i in indices[0]]
    return rerank(query, candidates, top_k=k_final)

# Example
result = hyde_rerank_pipeline("Where is the Eiffel Tower located?")
print("HyDE + Re-rank pipeline:")
for i, (doc, score) in enumerate(result, 1):
    print(f"  {i}. [{score:.3f}] {doc[:55]}...")

In [ ]:
def manual_relevance_score(contexts: list[str], query: str) -> list[tuple[str, int]]:
    """Manual evaluation: score each context 1-5 for relevance.
    In practice, you would collect human labels. Here we simulate with a heuristic."""
    def heuristic_score(ctx: str, q: str) -> int:
        q_words = set(q.lower().split())
        ctx_words = set(ctx.lower().split())
        overlap = len(q_words & ctx_words) / max(len(q_words), 1)
        if overlap >= 0.5:
            return 5
        if overlap >= 0.3:
            return 4
        if overlap >= 0.15:
            return 3
        if overlap > 0:
            return 2
        return 1
    return [(c, heuristic_score(c, query)) for c in contexts]

def context_precision(retrieved: list[tuple[str, int]]) -> float:
    """Average relevance of retrieved contexts (1-5 scale, normalized to 0-1)."""
    if not retrieved:
        return 0.0
    return sum(s for _, s in retrieved) / (5 * len(retrieved))

query = "Where is the Eiffel Tower?"
retrieved = ["The Eiffel Tower is in Paris.", "Python is popular for AI.", "Paris is the capital of France."]
scored = manual_relevance_score(retrieved, query)
print("Context relevance (1-5):")
for ctx, s in scored:
    print(f"  [{s}] {ctx[:50]}...")
print(f"\nContext precision: {context_precision(scored):.2f}")

In [ ]:
def answer_context_overlap(answer: str, context: str) -> float:
    """Simple faithfulness proxy: word overlap between answer and context."""
    a_words = set(answer.lower().split())
    c_words = set(context.lower().split())
    if not a_words:
        return 0.0
    return len(a_words & c_words) / len(a_words)

def faithfulness_score(answer: str, contexts: list[str]) -> float:
    """Average overlap of answer with each context (0-1)."""
    if not contexts:
        return 0.0
    overlaps = [answer_context_overlap(answer, c) for c in contexts]
    return max(overlaps)  # answer should align with at least one context

answer = "The Eiffel Tower is located in Paris, France."
contexts = ["The Eiffel Tower is in Paris.", "Paris is the capital of France."]
print("Answer:", answer)
print("Faithfulness (overlap with context):", faithfulness_score(answer, contexts))

# Hallucinated answer
bad_answer = "The Eiffel Tower is in London."
print("\nBad answer:", bad_answer)
print("Faithfulness:", faithfulness_score(bad_answer, contexts))

---
## 5. Debugging RAG Failures

**Diagnostic framework**: Is the problem in **retrieval** or **generation**?

- **Retrieval failure**: Wrong chunks, wrong granularity, semantic mismatch, missing info
- **Generation failure**: Good context but LLM ignores it or hallucinates

**Common failure patterns**:
- Chunks too small → missing context
- Chunks too large → noise dilutes relevance
- Query-document vocabulary mismatch → HyDE or multi-query helps
- Multi-hop questions → need iterative retrieval

In [ ]:
def rag_diagnostic(query: str, contexts: list[str], answer: str) -> dict:
    """Report retrieval scores, context relevance, and generation quality."""
    scored = manual_relevance_score(contexts, query)
    ctx_prec = context_precision(scored)
    faith = faithfulness_score(answer, contexts)
    return {
        "context_precision": ctx_prec,
        "faithfulness": faith,
        "context_scores": scored,
        "diagnosis": "Retrieval issue" if ctx_prec < 0.5 else ("Generation issue" if faith < 0.5 else "OK")
    }

# Example: good case
good = rag_diagnostic(
    "Where is the Eiffel Tower?",
    ["The Eiffel Tower is in Paris.", "Paris is the capital of France."],
    "The Eiffel Tower is in Paris."
)
print("Good case:", good["diagnosis"])
print("Context precision:", good["context_precision"])
print("Faithfulness:", good["faithfulness"])

# Example: retrieval failure (irrelevant context)
bad_ret = rag_diagnostic(
    "Where is the Eiffel Tower?",
    ["Python is popular.", "The brain has 86B neurons."],
    "I don't know."
)
print("\nRetrieval failure:", bad_ret["diagnosis"])
print("Context precision:", bad_ret["context_precision"])

# Example: generation failure (good context, bad answer)
bad_gen = rag_diagnostic(
    "Where is the Eiffel Tower?",
    ["The Eiffel Tower is in Paris."],
    "The Eiffel Tower is in London."
)
print("\nGeneration failure:", bad_gen["diagnosis"])
print("Faithfulness:", bad_gen["faithfulness"])

---
## 6. Exercises

1. **Exercise 1**: Implement HyDE for a domain of your choice (e.g., medical, legal, scientific) and compare retrieval quality with and without it. Use a small custom knowledge base.

2. **Exercise 2**: Build a re-ranking pipeline and measure **MRR (Mean Reciprocal Rank)** improvement. Create a small set of labeled query-document pairs (relevant/not) and compute MRR before and after re-ranking.

3. **Exercise 3**: Find a multi-hop question that your RAG system fails on with single retrieval, then fix it with iterative retrieval. Document the before/after behavior.

---
## 7. Responsible AI Reflection

> Advanced RAG techniques make systems more capable but also more opaque. When a system uses HyDE, multi-query, and re-ranking, it becomes harder to explain why a particular answer was generated. How do you balance retrieval sophistication with explainability? Should users be told which retrieval strategy was used?

**Discussion prompts:**
- What trade-offs exist between retrieval accuracy and interpretability?
- In high-stakes domains (healthcare, legal), how might you design for transparency?
- Could you surface "retrieval path" (e.g., which sub-queries led to which contexts) to users?

---
## 8. Summary & Next Steps

**Summary:**
- **Query transformation**: HyDE, multi-query, and query decomposition improve retrieval for complex or ambiguous queries
- **Re-ranking**: Cross-encoders improve precision after fast bi-encoder retrieval
- **Multi-hop**: Iterative retrieval handles questions that need information from multiple documents
- **Evaluation**: Context precision, faithfulness, and manual relevance scoring help diagnose RAG quality
- **Debugging**: Separate retrieval vs generation failures; use diagnostic functions to pinpoint issues

**Next steps:**
- Explore **RAGAS** (full framework) with OpenAI for production evaluation
- Implement **parent document retrieval** (retrieve small chunks, return larger parent chunks)
- Study **agentic RAG** — agents that decide when to retrieve, when to reason, and when to act